In [1]:
import openai, os
from openai import OpenAI
openai.api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=openai.api_key)

COMPLETION_MODEL = "gpt-3.5-turbo-instruct"
def generate_data_by_prompt(prompt):
    response = client.completions.create(
        model=COMPLETION_MODEL,
        prompt=prompt,
        temperature=0.5,
        max_tokens=2048,
        top_p=1,
    )
    
    return response.choices[0].text
prompt = """请你生成50条淘宝网里的商品的标题，每条在30个字左右，品类是3C数码产品，标题里往往也会有一些促销类的信息，每行一条。"""
data = generate_data_by_prompt(prompt)

ModuleNotFoundError: No module named 'openai'

In [7]:
import pandas as pd
product_names = data.strip().split('\n')
df = pd.DataFrame({'product_name': product_names})
df.head()

,product_name
0,1. 【限时秒杀】华为P30 Pro 麒麟980芯片 8GB+128GB 全网通 拍照手机
1,2. 【超值抢购】小米9 SE 全面屏智能手机 6GB+128GB 全网通 双卡双待
2,3. 【新品上市】Apple AirPods 无线蓝牙耳机 配备充电盒 支持Siri语音控制
3,4. 【限量发售】索尼WH-1000XM3 无线降噪耳机 高清音质 轻松旅行必备
4,5. 【限时特惠】华为平板M6 10.8英寸 4GB+64GB 全网通 高清大屏娱乐神器


In [8]:
df.product_name = df.product_name.apply(lambda x: x.split('.')[1].strip())
df.head()

,product_name
0,【限时秒杀】华为P30 Pro 麒麟980芯片 8GB+128GB 全网通 拍照手机
1,【超值抢购】小米9 SE 全面屏智能手机 6GB+128GB 全网通 双卡双待
2,【新品上市】Apple AirPods 无线蓝牙耳机 配备充电盒 支持Siri语音控制
3,【限量发售】索尼WH-1000XM3 无线降噪耳机 高清音质 轻松旅行必备
4,【限时特惠】华为平板M6 10


In [9]:
clothes_prompt = """请你生成50条淘宝网里的商品的标题，每条在30个字左右，品类是女性的服饰箱包等等，标题里往往也会有一些促销类的信息，每行一条。"""
clothes_data = generate_data_by_prompt(clothes_prompt)
clothes_product_names = clothes_data.strip().split('\n')
clothes_df = pd.DataFrame({'product_name': clothes_product_names})
clothes_df.product_name = clothes_df.product_name.apply(lambda x: x.split('.')[1].strip())
clothes_df.head()

,product_name
0,夏日必备！清新碎花连衣裙，轻松搭配，时尚百搭！
1,潮流女神必备！时尚牛仔短裤，修身显瘦，百变造型！
2,轻松出游，百搭小白鞋，舒适透气，时尚百搭！
3,清新甜美，碎花雪纺衫，柔软舒适，夏日必备！
4,时尚百搭，简约小黑裙，展现优雅气质，女神范儿！


In [10]:
df = pd.concat([df, clothes_df], axis=0)
df = df.reset_index(drop=True)
display(df)

,product_name
0,【限时秒杀】华为P30 Pro 麒麟980芯片 8GB+128GB 全网通 拍照手机
1,【超值抢购】小米9 SE 全面屏智能手机 6GB+128GB 全网通 双卡双待
2,【新品上市】Apple AirPods 无线蓝牙耳机 配备充电盒 支持Siri语音控制
3,【限量发售】索尼WH-1000XM3 无线降噪耳机 高清音质 轻松旅行必备
4,【限时特惠】华为平板M6 10
...,...
95,轻松出行，时尚帆布包，简约百搭，潮流必备！
96,精致女人必备！时尚半身裙，高贵典雅，气质满分！
97,夏日清凉，时尚露背上衣，性感迷人，吸睛必备！
98,时尚百搭，简约黑色短裤，修身显瘦，气质满分！


In [11]:
import openai, os, backoff

def get_embeddings(list_of_text, model):
    response = client.embeddings.create(input=list_of_text, model=model)
    return [item.embedding for item in response.data]
    
embedding_model = "text-embedding-ada-002"
batch_size = 100
@backoff.on_exception(backoff.expo, openai.RateLimitError)
def get_embeddings_with_backoff(prompts, model):
    embeddings = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        embeddings += get_embeddings(list_of_text=batch, model=model)
    return embeddings
    
prompts = df.product_name.tolist()
prompt_batches = [prompts[i:i+batch_size] for i in range(0, len(prompts), batch_size)]
embeddings = []
for batch in prompt_batches:
    batch_embeddings = get_embeddings_with_backoff(prompts=batch, model=embedding_model)
    embeddings += batch_embeddings
df["embedding"] = embeddings
df.to_parquet("data/taobao_product_title.parquet", index=False)


In [12]:
from embeddings_utils import get_embedding, cosine_similarity
# search through the reviews for a specific product
def search_product(df, query, n=3, pprint=True):
    product_embedding = get_embedding(
        query,
        model=embedding_model
    )
    df["similarity"] = df.embedding.apply(lambda x: cosine_similarity(x, product_embedding))
    results = (
        df.sort_values("similarity", ascending=False)
        .head(n)
        .product_name
    )
    # print(df)
    # print(results)
    if pprint:
        for r in results:
            print(r)
    return results
results = search_product(df, "自然淡雅背包", n=3)


轻松出游，时尚小挎包，多功能设计，便捷实用！
轻松出行，时尚小挎包，多功能设计，便捷实用！
轻松出行，时尚小挎包，多功能设计，便捷实用！


In [31]:
def recommend_product(df, product_name, n=3, pprint=True):
    product_embedding = df[df['product_name'] == product_name].iloc[0].embedding
    df["similarity"] = df.embedding.apply(lambda x: cosine_similarity(x, product_embedding))
    results = (
        df.sort_values("similarity", ascending=False)
        .head(n)
        .product_name
    )
    if pprint:
        for r in results:
            print(r)
    return results
results = recommend_product(df, "【限量发售】索尼WH-1000XM3 无线降噪耳机 高清音质 轻松旅行必备", n=3)

【限量发售】索尼WH-1000XM3 无线降噪耳机 高清音质 轻松旅行必备
【限量发售】索尼WH-1000XM4 无线降噪耳机 高清音质 轻松旅行必备
【疯狂抢购】索尼WH-1000XM3 无线降噪耳机 黑科技 助你隔绝外界噪音


In [2]:
import faiss
import numpy as np
import pandas as pd
def load_embeddings_to_faiss(df):
    embeddings = np.array(df['embedding'].tolist()).astype('float32')
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

# Read df from parquet, safe here as we don't want to 
# repeat the embedding calculation
df = pd.read_parquet('data/taobao_product_title.parquet')

index = load_embeddings_to_faiss(df)

In [14]:
# 检查索引的类型
index_type = type(index)
print("索引的类型：", index_type)
print("索引的属性和方法：", index.__dict__)

索引的类型： <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
索引的属性和方法： {'this': <Swig Object of type 'faiss::IndexFlatL2 *' at 0x1094edf20>}


In [20]:
def search_index(index, df, query, k=5):
    query_vector = np.array(get_embedding(query, model=embedding_model)).reshape(1, -1).astype('float32')
    distances, indexes = index.search(query_vector, k)
    # print(distances)
    
    # print(df)
    # print("------")
    # print(len(indexes))
    # print("-----------------------")
    results = []
    for i in range(len(indexes)):
        # print(indexes[i])
        # print("22222222222222")
        # print(df.iloc[indexes[i]])
        product_names = df.iloc[indexes[i]]['product_name'].values.tolist()
        # print(product_names)
        results.append((distances[i], product_names))    
    # print(results)
    # print(type(results))
    return results
products = search_index(index, df, "自然淡雅背包", k=3)
for distances, product_names in products:
    for i in range(len(distances)):
        print(product_names[i], distances[i])


轻松出游，时尚小挎包，多功能设计，便捷实用！ 0.2577033
轻松出行，时尚小挎包，多功能设计，便捷实用！ 0.26175284
轻松出行，时尚小挎包，多功能设计，便捷实用！ 0.26175284
